# Kpay Case Study - Merchant Data Analysis
## Business Development Strategy for Australian Merchants

## 1. Load & Inspect Data

In [40]:
#import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

#load dataset
FILE_PATH = "data/Case Study - Raw Data.csv"
df = pd.read_csv(FILE_PATH, dtype=str)  #preserves phone number formatting

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

Loaded: 199,999 rows × 10 columns

Columns: ['id', 'lead_key', 'phone', 'business_name', 'state', 'suburb', 'address', 'sector_level_1', 'sector_level_2', 'sector_level_3']


In [41]:
df.head()

,id,lead_key,phone,business_name,state,suburb,address,sector_level_1,sector_level_2,sector_level_3
0,2312630,L_611287243333,611287000000,PRO IT,NSW,sydney,"Suite 604, Level 6/83 York Street Sydney NSW 2000",Retail,Electronics & Appliances,"Computer, IT Technical Support"
1,2312631,L_611300048153,611300000000,Hoist Care,NSW,sydney,Sydney Sydney NSW 2000,Retail,Fashion & Accessories,"Wholesale Car Accessories, Manufacturers"
2,2312632,L_611300053384,611300000000,JEEVI,NSW,sydney,714/368 Sussex Street Sydney NSW 2000,Beauty & Wellness,Others,Community Health Services
3,2312633,L_611300069313,611300000000,Dunlap Bike Finance,NSW,sydney,377 Kent St Sydney NSW 2000,Retail,Fashion & Accessories,"Wholesale Motorcycle Parts and Accessories, Ma..."
4,2312634,L_611300074353,611300000000,Metiri Mensus,NSW,parramatta,Suite 107 L 1 30 Cowper Street Parramatta NSW...,F&B,Others,Food


In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 199999 entries, 0 to 199998
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   id              199999 non-null  str  
 1   lead_key        199999 non-null  str  
 2   phone           199999 non-null  str  
 3   business_name   199989 non-null  str  
 4   state           191664 non-null  str  
 5   suburb          94988 non-null   str  
 6   address         199945 non-null  str  
 7   sector_level_1  162229 non-null  str  
 8   sector_level_2  162229 non-null  str  
 9   sector_level_3  193153 non-null  str  
dtypes: str(10)
memory usage: 15.3 MB


In [43]:
df.describe(include='all')

,id,lead_key,phone,business_name,state,suburb,address,sector_level_1,sector_level_2,sector_level_3
count,199999,199999,199999,199989,191664,94988,199945,162229,162229,193153
unique,199999,199999,199665,189060,70,4393,175908,146,1554,7191
top,2312630,L_611287243333,611301000000,The Lott,NSW,Sydney,"Sydney,NSW,2000",Others,Others,Investing
freq,1,1,157,126,113341,3128,281,82508,91005,9918


In [44]:
print(f"\n=== DATA TYPES ===")
print(df.dtypes)


=== DATA TYPES ===
id                str
lead_key          str
phone             str
business_name     str
state             str
suburb            str
address           str
sector_level_1    str
sector_level_2    str
sector_level_3    str
dtype: object


## 2. Data Quality Check

In [45]:
print("=" * 55)
print("  DATA QUALITY CHECK — PART 1A")
print("=" * 55)

total = len(df)

# 1. Missing values
print("\n MISSING VALUES:")
missing = df.isnull().sum()
missing_pct = (missing / total * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# 2. Duplicates
dup_lead_key = df.duplicated(subset=['lead_key']).sum()
dup_name_suburb = df.duplicated(subset=['business_name', 'suburb']).sum()
print(f"\n DUPLICATES:")
print(f"  lead_key duplicates:              {dup_lead_key:,}")
print(f"  business_name + suburb dupes:     {dup_name_suburb:,}")

# 3. Phone analysis — THE KEY FINDING
print(f"\n PHONE ANALYSIS:")
print(f"  Total records:                    {total:,}")
print(f"\n  Top 20 most common phone values:")
print(df['phone'].value_counts().head(20).to_string())

# Masked phone patterns
masked_patterns = ['611300000000', '611287000000', '611800000000']
df['phone_masked'] = df['phone'].str.strip().isin(masked_patterns) | \
                     df['phone'].str.match(r'^6113000\d{5}$', na=False) | \
                     df['phone'].isna()

masked_count = df['phone_masked'].sum()
masked_pct = masked_count / total * 100
print(f"\n    Masked/invalid phones:          {masked_count:,} ({masked_pct:.1f}%)")
print(f"   Potentially valid phones:        {total - masked_count:,} ({100 - masked_pct:.1f}%)")

# 4. Sector breakdown
print(f"\n SECTOR BREAKDOWN (sector_level_1):")
print(df['sector_level_1'].value_counts().to_string())

# 5. Geographic spread
print(f"\n TOP 20 SUBURBS:")
print(df['suburb'].str.lower().value_counts().head(20).to_string())

# 6. State breakdown
print(f"\n STATE BREAKDOWN:")
print(df['state'].value_counts().to_string())

  DATA QUALITY CHECK — PART 1A

 MISSING VALUES:
                Missing Count  Missing %
business_name              10        0.0
state                    8335        4.2
suburb                 105011       52.5
address                    54        0.0
sector_level_1          37770       18.9
sector_level_2          37770       18.9
sector_level_3           6846        3.4

 DUPLICATES:
  lead_key duplicates:              0
  business_name + suburb dupes:     6,923

 PHONE ANALYSIS:
  Total records:                    199,999

  Top 20 most common phone values:
phone
611301000000       157
611300000000       105
611801000000        41
611800000000        31
610292000000         2
613832000000         2
613833000000         2
616386000000         2
611287000000         1
+61 408 268 946      1
+61 2 4939 3300      1
+61 459 484 837      1
(02) 5504 5491       1
0456 378 130         1
(02) 6581 0544       1
0400 745 366         1
+61 411 665 644      1
+61 414 500 589      1
0499 521 52

## 3. Business Impact Summary

In [46]:
print("=" * 55)
print("  BUSINESS IMPACT — BD COST OF DATA ISSUES")
print("=" * 55)

total = len(df)
masked_count = df['phone_masked'].sum()
dup_count = df.duplicated(subset=['lead_key']).sum()

# Non-target sectors
non_target_keywords = ['Retirement', 'Equestrian', 'NGO', 'Construction', 
                       'Community Health', 'Religious', 'Fabrication',
                       'Marriage Celebrant', 'Recycling']
non_target_mask = df['sector_level_3'].str.contains(
    '|'.join(non_target_keywords), case=False, na=False)
non_target_count = non_target_mask.sum()

# Others in sector_level_1
others_count = (df['sector_level_1'] == 'Others').sum()

print(f"""
┌─────────────────────────────────────────────────────────┐
│  ISSUE              │ COUNT      │ BD IMPACT             │
├─────────────────────────────────────────────────────────┤
│  Masked phones      │ {masked_count:>8,}   │ Uncontactable leads   │
│  Duplicate records  │ {dup_count:>8,}   │ Wasted repeat outreach│  
│  Non-target sectors │ {non_target_count:>8,}   │ Wrong pitch, low ROI  │
│  'Others' sector    │ {others_count:>8,}   │ Can't prioritise      │
└─────────────────────────────────────────────────────────┘

 KEY FINDING FOR SLIDE 3:
   Without cleaning: {masked_count + dup_count + non_target_count:,} records 
   ({((masked_count + dup_count + non_target_count)/total*100):.0f}%) would waste BD effort.
   
   Actionable Lead Rate BEFORE cleaning: 
   {((total - masked_count - non_target_count)/total*100):.1f}%
""")

  BUSINESS IMPACT — BD COST OF DATA ISSUES

┌─────────────────────────────────────────────────────────┐
│  ISSUE              │ COUNT      │ BD IMPACT             │
├─────────────────────────────────────────────────────────┤
│  Masked phones      │      137   │ Uncontactable leads   │
│  Duplicate records  │        0   │ Wasted repeat outreach│  
│  Non-target sectors │    2,394   │ Wrong pitch, low ROI  │
│  'Others' sector    │   82,508   │ Can't prioritise      │
└─────────────────────────────────────────────────────────┘

 KEY FINDING FOR SLIDE 3:
   Without cleaning: 2,531 records 
   (1%) would waste BD effort.

   Actionable Lead Rate BEFORE cleaning: 
   98.7%



## 4. Data Cleaning

In [47]:
import re
import phonenumbers
from phonenumbers import PhoneNumberType

print("=" * 55)
print("  DATA CLEANING — PART 1B")
print("=" * 55)

df_clean = df.copy()
steps = []

# ── Step 1: Remove lead_key duplicates ──
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['lead_key'], keep='first')
removed = before - len(df_clean)
steps.append(('Remove lead_key duplicates', before, len(df_clean), removed))
print(f" Step 1 — Dedup on lead_key: removed {removed:,} rows")

# ── Step 2: Standardise text fields ──
df_clean['business_name'] = df_clean['business_name'].str.strip().str.title()
df_clean['suburb'] = df_clean['suburb'].str.strip().str.lower()
df_clean['state'] = df_clean['state'].str.strip().str.upper()
df_clean['sector_level_1'] = df_clean['sector_level_1'].str.strip()
df_clean['sector_level_2'] = df_clean['sector_level_2'].str.strip()
df_clean['sector_level_3'] = df_clean['sector_level_3'].str.strip()
print(f" Step 2 — Standardised text casing across all fields")

# ── Step 3: Phone analysis with phonenumbers library ──

# 3a. Extract phone from lead_key
df_clean['phone_from_leadkey'] = df_clean['lead_key'].str.replace('L_', '', n=1)

# 3b. Parse & validate with phonenumbers
def classify_phone(raw_phone):
    """Parse a raw phone string, return (parsed_number, is_valid, is_possible, phone_type)."""
    if pd.isna(raw_phone):
        return None, False, False, None
    stripped = re.sub(r'[\s()\-+]', '', str(raw_phone))
    if not stripped:
        return None, False, False, None

    for attempt in [str(raw_phone), '+' + stripped, stripped]:
        try:
            parsed = phonenumbers.parse(attempt, 'AU')
            valid = phonenumbers.is_valid_number(parsed)
            possible = phonenumbers.is_possible_number(parsed)
            ptype = phonenumbers.number_type(parsed)
            return parsed, valid, possible, ptype
        except phonenumbers.NumberParseException:
            continue
    return None, False, False, None

# Apply to phone column
phone_results = df_clean['phone'].apply(classify_phone)
df_clean['_phone_parsed']   = phone_results.apply(lambda x: x[0])
df_clean['_phone_valid']    = phone_results.apply(lambda x: x[1])
df_clean['_phone_possible'] = phone_results.apply(lambda x: x[2])
df_clean['_phone_type_raw'] = phone_results.apply(lambda x: x[3])

# Apply to lead_key phone
lk_results = df_clean['phone_from_leadkey'].apply(classify_phone)
df_clean['_lk_parsed']   = lk_results.apply(lambda x: x[0])
df_clean['_lk_valid']    = lk_results.apply(lambda x: x[1])
df_clean['_lk_possible'] = lk_results.apply(lambda x: x[2])
df_clean['_lk_type_raw'] = lk_results.apply(lambda x: x[3])

# 3c. Map phone types to BD-friendly labels
# Note: AU 13xx short numbers are classified as SHARED_COST by phonenumbers
TYPE_MAP = {
    PhoneNumberType.MOBILE: 'mobile',
    PhoneNumberType.FIXED_LINE: 'landline',
    PhoneNumberType.FIXED_LINE_OR_MOBILE: 'landline',
    PhoneNumberType.TOLL_FREE: 'toll_free',
    PhoneNumberType.SHARED_COST: 'shared_cost',
    PhoneNumberType.PREMIUM_RATE: 'premium',
    PhoneNumberType.UAN: 'shared_cost',
}

def map_phone_type(ptype):
    if ptype is None:
        return 'unknown'
    return TYPE_MAP.get(ptype, 'unknown')

# Use lead_key type as primary (it has the real number), fall back to phone column
df_clean['phone_type'] = df_clean['_lk_type_raw'].apply(map_phone_type)
# Where lead_key type is unknown, try phone column
mask_unknown_lk = df_clean['phone_type'] == 'unknown'
df_clean.loc[mask_unknown_lk, 'phone_type'] = df_clean.loc[mask_unknown_lk, '_phone_type_raw'].apply(map_phone_type)

# 3d. Cross-reference lead_key vs phone column — match_score
def compute_match_score(row):
    phone_raw = re.sub(r'[\s()\-+]', '', str(row['phone'])) if pd.notna(row['phone']) else ''
    lk_raw = str(row['phone_from_leadkey']) if pd.notna(row['phone_from_leadkey']) else ''
    if not phone_raw or not lk_raw:
        return 0.0
    # Both are placeholder patterns
    placeholder_patterns = ['611300000000', '611800000000', '000000000000']
    if phone_raw in placeholder_patterns and lk_raw in placeholder_patterns:
        return 0.0
    if phone_raw == lk_raw:
        return 1.0
    # Check prefix match (masked trailing digits)
    min_len = min(len(phone_raw), len(lk_raw))
    if min_len >= 6 and phone_raw[:6] == lk_raw[:6]:
        # Trailing zeros in phone suggest masking
        if phone_raw.endswith('000000') or phone_raw.endswith('0000'):
            return 0.5
        return 0.5  # prefix match but different trailing
    return 0.2

df_clean['phone_match_score'] = df_clean.apply(compute_match_score, axis=1)

# 3e. Frequency analysis
phone_freq = df_clean['phone'].value_counts()
df_clean['phone_frequency'] = df_clean['phone'].map(phone_freq).fillna(0).astype(int)

# 3f. Pattern checks
def has_bad_pattern(phone_str):
    """Flag numbers with 6+ trailing zeros or all repeating digits."""
    if pd.isna(phone_str):
        return True
    s = re.sub(r'[\s()\-+]', '', str(phone_str))
    if not s:
        return True
    if re.search(r'0{6,}$', s):
        return True
    if len(set(s)) == 1 and len(s) > 3:
        return True
    return False

df_clean['_phone_bad_pattern'] = df_clean['phone'].apply(has_bad_pattern)
df_clean['_lk_bad_pattern'] = df_clean['phone_from_leadkey'].apply(has_bad_pattern)

# 3g. Assign phone_quality (Level 1 / Level 2 / Level 3)
def assign_phone_quality(row):
    is_valid = row['_phone_valid'] or row['_lk_valid']
    is_possible = row['_phone_possible'] or row['_lk_possible']
    match_score = row['phone_match_score']
    freq = row['phone_frequency']
    bad_pattern = row['_phone_bad_pattern'] and row['_lk_bad_pattern']

    # Level 1 (fake) conditions
    if bad_pattern:
        return 'Level 1'
    if match_score == 0.0:
        return 'Level 1'
    if freq > 10:
        return 'Level 1'
    if not is_valid and not is_possible:
        return 'Level 1'

    # Level 3 (valid) conditions
    if is_valid and match_score == 1.0 and freq <= 2 and not row['_phone_bad_pattern']:
        return 'Level 3'

    # Level 2 (suspicious) — everything else
    return 'Level 2'

df_clean['phone_quality'] = df_clean.apply(assign_phone_quality, axis=1)

# 3h. Assign reachability_score (0-30)
def assign_reachability(row):
    quality = row['phone_quality']
    ptype = row['phone_type']

    if quality == 'Level 1':
        return 0
    if quality == 'Level 2':
        return 3
    # Level 3
    score_map = {
        'mobile': 30,
        'landline': 25,
        'shared_cost': 20,
        'toll_free': 10,
        'premium': 0,
        'unknown': 15,
    }
    return score_map.get(ptype, 15)

df_clean['reachability_score'] = df_clean.apply(assign_reachability, axis=1)

# Drop internal helper columns
internal_cols = [c for c in df_clean.columns if c.startswith('_')]
df_clean.drop(columns=internal_cols, inplace=True)

# Summary
print(f"  Step 3 — Phone analysis (phonenumbers library):")
print(f"   phone_quality distribution:")
for level, count in df_clean['phone_quality'].value_counts().sort_index().items():
    print(f"     {level}: {count:>8,} ({count/len(df_clean)*100:.1f}%)")
print(f"   phone_type distribution:")
for ptype, count in df_clean['phone_type'].value_counts().items():
    print(f"     {ptype:>12s}: {count:>8,}")
print(f"   reachability_score: mean={df_clean['reachability_score'].mean():.1f}, median={df_clean['reachability_score'].median():.0f}")

# ── Step 4: Non-target filter ──
non_target_keywords = ['Retirement', 'Equestrian', 'NGO', 'Construction',
                       'Community Health', 'Religious', 'Fabrication',
                       'Marriage Celebrant', 'Recycling']
df_clean['kpay_relevant'] = ~df_clean['sector_level_3'].str.contains(
    '|'.join(non_target_keywords), case=False, na=False)
non_target_removed = (~df_clean['kpay_relevant']).sum()
print(f" Step 4 — Non-target filter: flagged {non_target_removed:,} irrelevant merchants")

# ── Step 5: Contactability / data completeness helpers ──
df_clean['has_address'] = df_clean['address'].notna()
df_clean['has_sector'] = df_clean['sector_level_1'].notna()
print(f" Step 5 — Data completeness flags created")

# ── Step 6: Flag name+suburb dupes ──
df_clean['name_suburb_dup'] = df_clean.duplicated(
    subset=['business_name', 'suburb'], keep='first')
dup_flagged = df_clean['name_suburb_dup'].sum()
print(f" Step 6 — Flagged {dup_flagged:,} name+suburb duplicates (kept, not deleted)")

  DATA CLEANING — PART 1B
 Step 1 — Dedup on lead_key: removed 0 rows
 Step 2 — Standardised text casing across all fields
  Step 3 — Phone analysis (phonenumbers library):
   phone_quality distribution:
     Level 1:      406 (0.2%)
     Level 2:  111,935 (56.0%)
     Level 3:   87,658 (43.8%)
   phone_type distribution:
         landline:  147,820
           mobile:   46,536
          unknown:    2,724
      shared_cost:    2,209
        toll_free:      710
   reachability_score: mean=13.4, median=3
 Step 4 — Non-target filter: flagged 2,394 irrelevant merchants
 Step 5 — Data completeness flags created
 Step 6 — Flagged 8,040 name+suburb duplicates (kept, not deleted)


## 5. Cleaning Funnel

In [48]:
# Working on relevant records only
df_relevant = df_clean[df_clean['kpay_relevant']].copy()

print("\n CLEANING FUNNEL:")
print(f"  Raw records:                      {len(df):>8,}")
print(f"  After dedup (lead_key):           {len(df_clean):>8,}")
print(f"  After non-target filter:          {len(df_relevant):>8,}")

# Phone quality breakdown
l3 = (df_relevant['phone_quality'] == 'Level 3').sum()
l2 = (df_relevant['phone_quality'] == 'Level 2').sum()
l1 = (df_relevant['phone_quality'] == 'Level 1').sum()
print(f"\n  Phone quality:")
print(f"    Level 3 (valid):                {l3:>8,} ({l3/len(df_relevant)*100:.1f}%)")
print(f"    Level 2 (suspicious):           {l2:>8,} ({l2/len(df_relevant)*100:.1f}%)")
print(f"    Level 1 (fake):                 {l1:>8,} ({l1/len(df_relevant)*100:.1f}%)")

# Reachability breakdown
contactable = (df_relevant['reachability_score'] > 0).sum()
print(f"\n  Contactable (reachability > 0):   {contactable:>8,} ({contactable/len(df_relevant)*100:.1f}%)")
print(f"  With address:                     {df_relevant['has_address'].sum():>8,}")

actionable = l3 + l2  # Level 2+ are worth attempting
actionable_pct = actionable / len(df) * 100
print(f"""
  ACTIONABLE LEAD RATE AFTER CLEANING: {actionable_pct:.1f}%
   These are Level 2+ phone quality in KPay-relevant sectors.
   Level 3 (high confidence): {l3:,} | Level 2 (worth trying): {l2:,}
""")


 CLEANING FUNNEL:
  Raw records:                       199,999
  After dedup (lead_key):            199,999
  After non-target filter:           197,605

  Phone quality:
    Level 3 (valid):                  87,148 (44.1%)
    Level 2 (suspicious):            110,054 (55.7%)
    Level 1 (fake):                      403 (0.2%)

  Contactable (reachability > 0):    197,202 (99.8%)
  With address:                      197,553

  ACTIONABLE LEAD RATE AFTER CLEANING: 98.6%
   These are Level 2+ phone quality in KPay-relevant sectors.
   Level 3 (high confidence): 87,148 | Level 2 (worth trying): 110,054



## 6. Cleaning and filtering Australia and others

In [50]:
print("=" * 55)
print("  SMART COUNTRY DETECTION")
print("=" * 55)

import re

# ══════════════════════════════════════════════════════
# REFERENCE DICTIONARIES
# ══════════════════════════════════════════════════════

AU_STATE_MAP = {
    'NSW': 'NSW', 'VIC': 'VIC', 'QLD': 'QLD',
    'WA': 'WA',   'SA': 'SA',   'ACT': 'ACT',
    'TAS': 'TAS', 'NT': 'NT',   'AU': 'AU',
    'New South Wales': 'NSW',
    'Victoria': 'VIC',
    'Queensland': 'QLD',
    'Western Australia': 'WA',
    'South Australia': 'SA',
    'Australian Capital Territory': 'ACT',
    'Tasmania': 'TAS',
    'Northern Territory': 'NT',
    'Sydney, Nsw': 'NSW',
    'Sydney, NSW': 'NSW',
}

US_STATES = {
    'KY','NY','CA','FL','NC','TX','OR','MA','GA','CT','PA',
    'OH','MD','AZ','VT','SC','IL','MI','TN','CO','ME','KS',
    'UT','IN','RI','ID','ND','NJ','NH','AL','WI','NV','OK',
    'SD','HI','MN','MO','NM','DC','WV','VA','MT','DE'
}

GARBAGE_VALUES = {
    'APAC','HOP','AMM','ATM','LOT','PTD','STE',
    'BPM','NEX','ATT','GD','IGO','BG'
}

AU_ADDRESS_SIGNALS = [
    r'\bNSW\b', r'\bVIC\b', r'\bQLD\b', r'\bWA\b',
    r'\bSA\b',  r'\bACT\b', r'\bTAS\b', r'\bNT\b',
    r'New South Wales', r'Victoria', r'Queensland',
    r'Western Australia', r'South Australia',
    r'Australian Capital Territory', r'Tasmania',
    r'Northern Territory', r'Australia',
    r'\bSydney\b', r'\bMelbourne\b', r'\bBrisbane\b',
    r'\bPerth\b', r'\bAdelaide\b', r'\bCanberra\b',
    r'\bHobart\b', r'\bDarwin\b',
]

OVERSEAS_SIGNALS = [
    r'United Kingdom', r'United States', r'Canada',
    r'New Zealand', r'South Africa', r'France',
    r'Germany', r'China', r'Japan', r'Singapore',
    r'\b[A-Z]{1,2}\d{1,2}[A-Z]?\s\d[A-Z]{2}\b',  # UK postcode
    r'\bQC\b', r'\bON\b', r'\bBC\b', r'\bAB\b',   # Canadian provinces
    r'\bMB\b', r'\bNL\b', r'\bNS\b', r'\bNB\b',
    r'\b[A-Z]{2}\s\d{5}\b',                         # US ZIP pattern
    r'\bRue\b', r'\bBoulevard\b.*\bLaval\b',        # French patterns
]

# ══════════════════════════════════════════════════════
# CLASSIFICATION FUNCTION — returns (is_au, state_clean)
# ══════════════════════════════════════════════════════

def classify_record(row):
    state_raw = str(row['state']).strip() if pd.notna(row['state']) else ''
    address_raw = str(row['address']).strip() if pd.notna(row['address']) else ''

    state_titled = state_raw.title()
    state_upper  = state_raw.upper()

    # Step 1: Known AU state → confirmed AU
    if state_titled in AU_STATE_MAP:
        return (True, AU_STATE_MAP[state_titled])
    if state_upper in AU_STATE_MAP:
        return (True, AU_STATE_MAP[state_upper])

    # Step 2: Known US state → confirmed overseas
    if state_upper in US_STATES:
        return (False, None)

    # Step 3: State is null, garbage, or unknown
    #         → interrogate address

    # Check overseas signals first (more specific)
    for signal in OVERSEAS_SIGNALS:
        if re.search(signal, address_raw, re.IGNORECASE):
            return (False, None)

    # Check AU signals
    for signal in AU_ADDRESS_SIGNALS:
        if re.search(signal, address_raw, re.IGNORECASE):
            return (True, 'AU_rescued')

    # Step 4: No signal found → uncertain → goes to out_of_AU
    return (False, None)

# ══════════════════════════════════════════════════════
# APPLY TO FULL DATASET
# ══════════════════════════════════════════════════════

print("\n Classifying all records...")
results = df_clean.apply(classify_record, axis=1)

df_clean['is_au']       = results.apply(lambda x: x[0])
df_clean['state_clean'] = results.apply(lambda x: x[1])

# ══════════════════════════════════════════════════════
# SPLIT INTO 2 FILES
# ══════════════════════════════════════════════════════

df_au       = df_clean[df_clean['is_au'] == True].copy()
df_non_au   = df_clean[df_clean['is_au'] == False].copy()

# Drop helper column — not needed in output files
df_au.drop(columns=['is_au'], inplace=True)
df_non_au.drop(columns=['is_au'], inplace=True)

# ══════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════

rescued_count = (df_au['state_clean'] == 'AU_rescued').sum()

print(f"\n CLASSIFICATION RESULTS:")
print(f"    AU records (confirmed):   {(df_au['state_clean'] != 'AU_rescued').sum():>8,}")
print(f"    AU records (rescued):     {rescued_count:>8,}  ← had null/bad state but AU address")
print(f"    Non-AU + uncertain:       {len(df_non_au):>8,}  → Out_of_AU_Data.csv")
print(f"    Total:                   {len(df_au)+len(df_non_au):>8,}")

print(f"\n AU STATE DISTRIBUTION (after cleaning):")
print(df_au['state_clean'].value_counts().to_string())

print(f"\n SAMPLE OF RESCUED AU RECORDS:")
rescued_sample = df_au[df_au['state_clean'] == 'AU_rescued']
print(rescued_sample[['business_name','state','suburb','address']].head(5).to_string())

print(f"\n SAMPLE OF NON-AU RECORDS:")
print(df_non_au[['business_name','state','address']].head(5).to_string())

# ══════════════════════════════════════════════════════
# EXPORT 2 FILES
# ══════════════════════════════════════════════════════

df_au.to_csv("data/Cleaned_AU_Data.csv", index=False)
df_non_au.to_csv("data/Out_of_AU_Data.csv", index=False)

print(f"\n Exported: data/Cleaned_AU_Data.csv    ({len(df_au):,} rows)")
print(f" Exported: data/Out_of_AU_Data.csv     ({len(df_non_au):,} rows)")

# ══════════════════════════════════════════════════════
# UPDATE df_relevant FOR ALL DOWNSTREAM CELLS
# ══════════════════════════════════════════════════════

df_relevant = df_au[df_au['kpay_relevant']].copy()
print(f"\n df_relevant updated: {len(df_relevant):,} AU + KPay-relevant records")


  SMART COUNTRY DETECTION

 Classifying all records...



 CLASSIFICATION RESULTS:
    AU records (confirmed):    191,042
    AU records (rescued):          138  ← had null/bad state but AU address
    Non-AU + uncertain:          8,819  → Out_of_AU_Data.csv
    Total:                    199,999

 AU STATE DISTRIBUTION (after cleaning):
state_clean
NSW           113865
VIC            33321
QLD            21753
WA              9660
SA              7833
ACT             2313
TAS             1708
NT               588
AU_rescued       138
AU                 1

 SAMPLE OF RESCUED AU RECORDS:
              business_name state     suburb                                                                        address
982                 Detoxfi   AMM        NaN  AMM 116452, AIC 56a Anzac st 2nd Floor, Suite 5, Chullora NSW 2190, Australia
2292   Goodskin Plus Beauty   NaN  brookvale                Australia, New South Wales, Brookvale, Condamine St, 邮政编码: 2100
5879            Mr Vitamins   NaN        NaN                                                

## 7. Filling up the null suburb from address

In [51]:
print("=" * 55)
print("  SUBURB EXTRACTION FROM ADDRESS")
print("=" * 55)

import re
import urllib.request
import io

# ══════════════════════════════════════════════════════
# STEP 1: LOAD AU SUBURB REFERENCE LIST
# ══════════════════════════════════════════════════════

print("\n Loading AU suburb reference list...")

url = "https://raw.githubusercontent.com/matthewproctor/australianpostcodes/master/australian_postcodes.csv"

try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    ref_df = pd.read_csv(io.StringIO(response.read().decode('utf-8')), dtype=str)
    au_suburbs = set(ref_df['locality'].str.lower().str.strip().dropna().unique())
    print(f" Loaded {len(au_suburbs):,} official AU suburbs")
except Exception as e:
    print(f"  Could not load from URL: {e}")
    print("   Falling back to dataset's own suburb values...")
    # Best fallback: use suburbs already present in the dataset itself
    au_suburbs = set(
        df_au['suburb'].str.lower().str.strip().dropna().unique()
    )
    print(f"   Using {len(au_suburbs):,} suburbs extracted from existing data")

# ══════════════════════════════════════════════════════
# STEP 2: DEFINE EXTRACTION FUNCTION
# ══════════════════════════════════════════════════════

# All AU state patterns — longer ones first to avoid partial matches
STATE_PATTERNS = [
    'New South Wales', 'Victoria', 'Queensland',
    'Western Australia', 'South Australia',
    'Australian Capital Territory', 'Tasmania',
    'Northern Territory',
    'NSW', 'VIC', 'QLD', 'WA', 'SA', 'ACT', 'TAS', 'NT'
]

# Words that are never suburbs — skip these during extraction
SKIP_TOKENS = {
    'shop', 'suite', 'level', 'floor', 'unit', 'lot',
    'kiosk', 'ground', 'building', 'tower', 'block',
    'cnr', 'corner', 'and', 'the', 'at', 'of', 'in',
    'australia', 'australian',
    'road', 'rd', 'street', 'st', 'ave', 'avenue',
    'drive', 'dr', 'lane', 'ln', 'place', 'pl',
    'court', 'ct', 'parade', 'pde', 'highway', 'hwy',
    'way', 'close', 'cl', 'crescent', 'cr', 'cres',
    'boulevard', 'blvd', 'terrace', 'tce', 'grove', 'gr',
    'location', 'store', 'office', 'centre', 'center',
    'mall', 'plaza', 'arcade', 'complex', 'park',
    'north', 'south', 'east', 'west',  # only skip standalone
}

def extract_suburb_from_address(address_str):
    """
    Extracts suburb from AU address string by:
    1. Finding state code position in address
    2. Extracting tokens before the state code
    3. Skipping postcodes and non-suburb tokens
    4. Trying 3-word, 2-word, 1-word candidates
    5. Cross-validating each against official AU suburb list
    Returns lowercase suburb string, or None if no confident match
    """
    if pd.isna(address_str) or not str(address_str).strip():
        return None

    addr = str(address_str).strip()

    # ── Find state code position ──
    state_pos = None
    for state in STATE_PATTERNS:
        match = re.search(r'\b' + re.escape(state) + r'\b', addr, re.IGNORECASE)
        if match:
            state_pos = match.start()
            break

    if state_pos is None:
        return None  # No state found → can't extract reliably

    # ── Get text before state code ──
    text_before = addr[:state_pos].strip()

    # ── Tokenise by comma, space, slash, dash ──
    raw_tokens = re.split(r'[\s,/\-]+', text_before)

    # ── Clean tokens ──
    clean_tokens = []
    for t in raw_tokens:
        t = t.strip('.,;:()')
        if not t:
            continue
        # Skip 4-digit postcodes
        if re.match(r'^\d{4}$', t):
            continue
        # Skip street numbers e.g. "19a", "260A", "3"
        if re.match(r'^\d+[A-Za-z]?$', t):
            continue
        # Skip non-suburb words
        if t.lower() in SKIP_TOKENS:
            continue
        clean_tokens.append(t)

    if not clean_tokens:
        return None

    # ── Try 3-word, 2-word, 1-word candidates ──
    # Always try longest match first → correctly handles
    # "Bondi Junction", "North Sydney", "Bella Vista" etc.
    for n in [3, 2, 1]:
        if len(clean_tokens) >= n:
            # Take LAST n tokens — closest to state code
            candidate = ' '.join(clean_tokens[-n:]).lower().strip()
            if candidate in au_suburbs:
                return candidate

    # No validated match found → don't guess
    return None

# ══════════════════════════════════════════════════════
# STEP 3: APPLY TO NULL SUBURBS ONLY
# ══════════════════════════════════════════════════════

print("\n Extracting suburbs from address column...")

null_suburb_mask = df_au['suburb'].isna()
null_count_before = null_suburb_mask.sum()
print(f"   Null suburbs before: {null_count_before:,}")

# Apply extraction only where suburb is null
extracted = df_au.loc[null_suburb_mask, 'address'].apply(
    extract_suburb_from_address
)

# Fill extracted values back into df_au
df_au.loc[null_suburb_mask, 'suburb'] = extracted

# ══════════════════════════════════════════════════════
# STEP 4: SUMMARY
# ══════════════════════════════════════════════════════

null_count_after = df_au['suburb'].isna().sum()
filled_count = null_count_before - null_count_after
fill_rate = filled_count / null_count_before * 100

print(f"\n SUBURB EXTRACTION RESULTS:")
print(f"   Null suburbs before:     {null_count_before:>8,}")
print(f"   Successfully filled:     {filled_count:>8,}  ({fill_rate:.1f}% fill rate)")
print(f"   Still null after:        {null_count_after:>8,}  (no extractable suburb in address)")
print(f"   Total non-null suburbs:  {df_au['suburb'].notna().sum():>8,}")

# Quality check — sample of filled records
print(f"\n SAMPLE — EXTRACTED SUBURBS (first 8):")
filled_sample = df_au[
    null_suburb_mask & df_au['suburb'].notna()
][['business_name', 'address', 'suburb']].head(8)
print(filled_sample.to_string())

# Show cases that still failed — understand why
print(f"\n SAMPLE — STILL NULL AFTER EXTRACTION (first 5):")
still_null = df_au[
    null_suburb_mask & df_au['suburb'].isna()
][['business_name', 'address', 'suburb']].head(5)
print(still_null.to_string())

# Top suburbs after filling
print(f"\n TOP 20 SUBURBS AFTER EXTRACTION:")
print(df_au['suburb'].str.lower().value_counts().head(20).to_string())

# ══════════════════════════════════════════════════════
# STEP 5: EXPORT + UPDATE df_relevant
# ══════════════════════════════════════════════════════

df_au.to_csv("data/Cleaned_AU_Data.csv", index=False)
print(f"\n data/Cleaned_AU_Data.csv updated with extracted suburbs")

df_relevant = df_au[df_au['kpay_relevant']].copy()
print(f" df_relevant updated: {len(df_relevant):,} records")

  SUBURB EXTRACTION FROM ADDRESS

 Loading AU suburb reference list...
 Loaded 16,220 official AU suburbs

 Extracting suburbs from address column...
   Null suburbs before: 96,196

 SUBURB EXTRACTION RESULTS:
   Null suburbs before:       96,196
   Successfully filled:       91,646  (95.3% fill rate)
   Still null after:           4,550  (no extractable suburb in address)
   Total non-null suburbs:   186,630

 SAMPLE — EXTRACTED SUBURBS (first 8):
              business_name                                                                                                                                                                                                              address    suburb
982                 Detoxfi                                                                                                                                        AMM 116452, AIC 56a Anzac st 2nd Floor, Suite 5, Chullora NSW 2190, Australia  chullora
1082   Falak Indian Cuisine                  

## 8. KMF Scoriing

In [52]:
print("=" * 55)
print("  SCORE KMF ON CLEAN AU DATASET")
print("=" * 55)

# ── At this point df_au has: ──
#  AU-only records 
#  Filled suburbs 
#  Phone enrichment columns from cleaning cell
#  Missing: KMF scoring columns 

df_final = df_au[df_au['kpay_relevant']].copy()
print(f"\n Working dataset: {len(df_final):,} AU + KPay-relevant records")

# ══════════════════════════════════════════════════════
# REACHABILITY (40 pts)
# reachability_score (0-30) from phone + has_address (0-10)
# ══════════════════════════════════════════════════════
df_final['reach_address'] = df_final['has_address'].astype(int) * 10
df_final['reachability']  = df_final['reachability_score'] + df_final['reach_address']

# ══════════════════════════════════════════════════════
# SECTOR PRIORITY (35 pts)
# ══════════════════════════════════════════════════════
sector_map = {
    'F&B': 35,
    'Beauty & Wellness': 26,
    'Retail': 26,
    'Professional Services': 12,
    'Others': 9
}
df_final['sector_score'] = df_final['sector_level_1'].map(sector_map).fillna(6)

# ══════════════════════════════════════════════════════
# GEO EFFICIENCY (15 pts) — now uses FILLED suburbs
# ══════════════════════════════════════════════════════
tier_a = [
    'hurstville','chatswood','burwood','parramatta','cabramatta',
    'eastwood','ashfield','strathfield','campsie','haymarket'
]
tier_b = [
    'bankstown','kogarah','roselands','rhodes','merrylands',
    'auburn','lidcombe','granville'
]

def geo_score(suburb):
    if pd.isna(suburb):
        return 1
    s = str(suburb).lower().strip()
    if s in tier_a:
        return 15
    elif s in tier_b:
        return 10
    elif 'sydney' in s:
        return 5
    else:
        return 1

df_final['geo_score'] = df_final['suburb'].apply(geo_score)

# ══════════════════════════════════════════════════════
# DATA COMPLETENESS (10 pts)
# ══════════════════════════════════════════════════════
df_final['data_completeness'] = (
    df_final['has_address'].astype(int) * 3 +
    df_final['has_sector'].astype(int) * 3 +
    (df_final['phone_match_score'] * 4).round().astype(int)
)

# ══════════════════════════════════════════════════════
# FINAL KMF SCORE
# ══════════════════════════════════════════════════════
df_final['kmf_score'] = (
    df_final['reachability'] +
    df_final['sector_score'] +
    df_final['geo_score'] +
    df_final['data_completeness']
)

def assign_tier(score):
    if score >= 70: return 'Tier 1'
    elif score >= 40: return 'Tier 2'
    else: return 'Tier 3'

df_final['tier'] = df_final['kmf_score'].apply(assign_tier)

# ══════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════
print(f"\n KMF BUDGET: Reachability(40) + Sector(35) + Geo(15) + Completeness(10) = 100")

print(f"\n KMF SCORE DISTRIBUTION:")
print(f"   Mean:   {df_final['kmf_score'].mean():.1f}")
print(f"   Median: {df_final['kmf_score'].median():.0f}")
print(f"   Std:    {df_final['kmf_score'].std():.1f}")
print(f"   Min:    {df_final['kmf_score'].min():.0f}")
print(f"   Max:    {df_final['kmf_score'].max():.0f}")

print(f"\n TIER BREAKDOWN:")
tier_summary = df_final.groupby('tier').agg(
    Count=('kmf_score','count'),
    Avg_KMF=('kmf_score','mean'),
    Min_KMF=('kmf_score','min'),
    Max_KMF=('kmf_score','max')
).round(1)
print(tier_summary)

print(f"\n COMPONENT DISTRIBUTIONS:")
for comp in ['reachability','sector_score','geo_score','data_completeness']:
    s = df_final[comp]
    print(f"   {comp:>20s}: mean={s.mean():.1f}, min={s.min():.0f}, max={s.max():.0f}")

print(f"\n TIER 1 — TOP SECTORS:")
print(df_final[df_final['tier']=='Tier 1']['sector_level_1'].value_counts().head(8))

print(f"\n TIER 1 — TOP SUBURBS:")
print(df_final[df_final['tier']=='Tier 1']['suburb'].value_counts().head(10))

print(f"\n COLUMN CHECK:")
print(f"   Total columns: {len(df_final.columns)}")
print(f"   Columns: {list(df_final.columns)}")

# ══════════════════════════════════════════════════════
# EXPORT FINAL FILE — overwrites Cleaned_AU_Data.csv
# ══════════════════════════════════════════════════════
df_final.to_csv("data/Cleaned_AU_Data.csv", index=False)
print(f"\n FINAL EXPORT: data/Cleaned_AU_Data.csv")
print(f"   Rows:    {len(df_final):,}")
print(f"   Columns: {len(df_final.columns)}")

# Update df_relevant to point to final dataset
df_relevant = df_final.copy()

  SCORE KMF ON CLEAN AU DATASET

 Working dataset: 188,789 AU + KPay-relevant records

 KMF BUDGET: Reachability(40) + Sector(35) + Geo(15) + Completeness(10) = 100

 KMF SCORE DISTRIBUTION:
   Mean:   47.1
   Median: 47
   Std:    17.6
   Min:    24
   Max:    100

 TIER BREAKDOWN:
        Count  Avg_KMF  Min_KMF  Max_KMF
tier                                    
Tier 1  29321     78.9     70.0    100.0
Tier 2  76313     52.8     40.0     69.0
Tier 3  83155     30.7     24.0     39.0

 COMPONENT DISTRIBUTIONS:
           reachability: mean=23.9, min=10, max=40
           sector_score: mean=13.3, min=6, max=35
              geo_score: mean=2.1, min=1, max=15
      data_completeness: mean=7.8, min=4, max=10

 TIER 1 — TOP SECTORS:
sector_level_1
Retail                   12922
F&B                      10086
Beauty & Wellness         5871
Professional Services      264
Others                     140
Restaurant                   5
Health & Beauty              4
Retail Shopping              

In [ ]:
#pip install rapidfuzz

In [ ]:
#pip install --upgrade pip

## 9. Remapping the sectors to improve the score of KMF and accurate their performance

In [53]:
print("=" * 55)
print(" SMART SECTOR REMAPPING")
print("=" * 55)

from rapidfuzz import fuzz

# ══════════════════════════════════════════════════════
# SEED KEYWORDS — drives 90% of remapping
# ══════════════════════════════════════════════════════

SEED_MAP = {
    'F&B': [
        'food', 'restaurant', 'cafe', 'grocery', 'drink',
        'beverage', 'dining', 'catering', 'coffee', 'bistro',
        'seafood', 'bakery', 'pizza', 'sushi', 'dessert',
        'ice cream', 'gelato', 'juice', 'bubble tea', 'bar',
        'pub', 'grill', 'burger', 'noodle', 'dumpling', 'ramen',
        'takeaway', 'deli', 'butcher', 'liquor', 'bottle shop',
        'fruit', 'vegetable', 'supermarket', 'snack',
    ],
    'Beauty & Wellness': [
        'beauty', 'hair', 'nail', 'salon', 'spa', 'massage',
        'barber', 'skin', 'wellness', 'cosmetic', 'wax',
        'lash', 'tattoo', 'piercing', 'makeup', 'manicure',
        'pedicure', 'facial', 'tanning', 'hairdress', 'blowdry',
        'yoga', 'pilates', 'fitness', 'gym', 'physio',
        'chiropractic', 'acupuncture', 'health clinic',
    ],
    'Retail': [
        'clothing', 'fashion', 'apparel', 'shoe', 'jewel',
        'furniture', 'gift', 'toy', 'book', 'flower', 'florist',
        'sport', 'bicycle', 'baby', 'pet', 'market', 'homewares',
        'stationery', 'laundry', 'dry clean', 'tailor',
        'tobacco', 'perfume', 'fragrance', 'variety', 'discount',
        'car accessories', 'automotive parts', 'tyre', 'watch',
        'decoration', 'craft', 'hobby', 'outdoor', 'camping',
    ],
    'Retail - Electronics': [
        'electronic', 'computer', 'laptop', 'technology',
        'appliance', 'audio', 'printer', 'gaming', 'software',
        'hardware', 'mobile', 'tablet', 'electrical', 'camera',
        'musical instrument', 'it support', 'phone repair',
        'kitchen appliance', 'office equipment',
    ],
    'Professional Services': [
        'repair', 'medical', 'dental', 'legal', 'lawyer',
        'accounting', 'consult', 'cleaning', 'education',
        'travel', 'real estate', 'insurance', 'financial',
        'mortgage', 'immigration', 'mechanic', 'plumber',
        'electrician', 'removalist', 'logistics', 'pharmacy',
        'doctor', 'optometrist', 'psychology', 'counselling',
        'printing', 'signage', 'photography', 'childcare',
    ],
}

TARGET_SECTORS = list(SEED_MAP.keys())

# ══════════════════════════════════════════════════════
# REMAPPING FUNCTION
# ══════════════════════════════════════════════════════

def smart_remap(sector_val, l3_val=None):
    """
    Remaps sector_level_1 to clean category using:
    1. Seed keyword match on sector_level_1
    2. Seed keyword match on sector_level_3 (for Others/null)
    3. Fuzzy match on sector_level_1 as last resort
    Returns clean category or original value if no confident match
    """
    is_others_or_null = (
        pd.isna(sector_val) or
        str(sector_val).strip() in ['Others', 'nan', '']
    )

    val = str(sector_val).lower().strip() if pd.notna(sector_val) else ''

    # ── Step 1: Seed keyword on sector_level_1 ──
    if not is_others_or_null:
        for category, keywords in SEED_MAP.items():
            if any(kw in val for kw in keywords):
                return category

    # ── Step 2: For Others/null → try sector_level_3 ──
    if is_others_or_null and pd.notna(l3_val):
        l3 = str(l3_val).lower().strip()
        for category, keywords in SEED_MAP.items():
            if any(kw in l3 for kw in keywords):
                return category

    # ── Step 3: Fuzzy match on sector_level_1 (non-Others only) ──
    if not is_others_or_null and val:
        scores = {
            cat: fuzz.partial_ratio(val, cat.lower())
            for cat in TARGET_SECTORS
        }
        best_cat  = max(scores, key=scores.get)
        best_score = scores[best_cat]
        if best_score >= 75:
            return best_cat

    # ── No confident match ──
    return sector_val if not is_others_or_null else 'Others'


# ══════════════════════════════════════════════════════
# APPLY TO DATASET
# ══════════════════════════════════════════════════════

print("\n Remapping sectors...")

before_counts = df_final['sector_level_1'].value_counts()

df_final['sector_level_1_clean'] = df_final.apply(
    lambda row: smart_remap(row['sector_level_1'], row['sector_level_3']),
    axis=1
)

# ══════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════

after_counts = df_final['sector_level_1_clean'].value_counts()

print(f"\n SECTOR DISTRIBUTION BEFORE vs AFTER:")
print(f"\n   {'Category':<25} {'Before':>8}  {'After':>8}  {'Change':>8}")
print(f"   {'-'*58}")

report_cats = TARGET_SECTORS + ['Others']
for cat in report_cats:
    before = before_counts.get(cat, 0)
    after  = after_counts.get(cat, 0)
    change = after - before
    sign   = '+' if change >= 0 else ''
    print(f"   {cat:<25} {before:>8,}  {after:>8,}  {sign}{change:>7,}")

null_before = df_final['sector_level_1'].isna().sum()
null_after  = df_final['sector_level_1_clean'].isna().sum()
print(f"   {'Null':<25} {null_before:>8,}  {null_after:>8,}  {null_before - null_after:>+8,}")

rescued = (
    (df_final['sector_level_1'].isna() |
     (df_final['sector_level_1'] == 'Others')) &
    (df_final['sector_level_1_clean'].isin(TARGET_SECTORS))
).sum()
print(f"\n    Rescued from Others/null: {rescued:,} records")

# Show what's still unmapped — good to know
remaining_unique = df_final[
    ~df_final['sector_level_1_clean'].isin(TARGET_SECTORS + ['Others'])
    & df_final['sector_level_1_clean'].notna()
]['sector_level_1_clean'].value_counts().head(10)

if len(remaining_unique) > 0:
    print(f"\n TOP UNMAPPED LABELS (still non-standard):")
    print(remaining_unique.to_string())

# ══════════════════════════════════════════════════════
# UPDATE SECTOR SCORE + RECALCULATE KMF
# ══════════════════════════════════════════════════════

sector_score_map = {
    'F&B': 35,
    'Beauty & Wellness': 26,
    'Retail': 26,
    'Professional Services': 12,
    'Retail - Electronics': 12,
    'Others': 9,
}

df_final['sector_score'] = df_final['sector_level_1_clean'].map(
    sector_score_map).fillna(6)

df_final['kmf_score'] = (
    df_final['reachability'] +
    df_final['sector_score'] +
    df_final['geo_score'] +
    df_final['data_completeness']
)

df_final['tier'] = df_final['kmf_score'].apply(assign_tier)

print(f"\n UPDATED TIER BREAKDOWN (after sector remapping):")
tier_summary = df_final.groupby('tier').agg(
    Count=('kmf_score', 'count'),
    Avg_KMF=('kmf_score', 'mean'),
    Min_KMF=('kmf_score', 'min'),
    Max_KMF=('kmf_score', 'max')
).round(1)
print(tier_summary)

print(f"\n TIER 1 — TOP SECTORS (clean labels):")
print(df_final[df_final['tier']=='Tier 1']['sector_level_1_clean'].value_counts().head(6))

print(f"\n TIER 1 — TOP SUBURBS:")
print(df_final[df_final['tier']=='Tier 1']['suburb'].value_counts().head(10))

# ══════════════════════════════════════════════════════
# EXPORT
# ══════════════════════════════════════════════════════

df_final.to_csv("data/Cleaned_AU_Data.csv", index=False)
print(f"\n Exported: data/Cleaned_AU_Data.csv")
print(f"   Rows: {len(df_final):,}  |  Columns: {len(df_final.columns)}")

df_relevant = df_final.copy()
print(f"\n smart sector remapping done")

 SMART SECTOR REMAPPING

 Remapping sectors...

 SECTOR DISTRIBUTION BEFORE vs AFTER:

   Category                    Before     After    Change
   ----------------------------------------------------------
   F&B                         12,517    27,010  + 14,493
   Beauty & Wellness            9,377    24,855  + 15,478
   Retail                      24,324    40,395  + 16,071
   Retail - Electronics             0     2,895  +  2,895
   Professional Services       16,602    35,258  + 18,656
   Others                      79,659    57,990  -21,669
   Null                        37,422         0   +37,422

    Rescued from Others/null: 59,091 records

 TOP UNMAPPED LABELS (still non-standard):
sector_level_1_clean
Stores                     145
Footwear                    44
Groceries                   34
Glassware & Earthenware     26
Office Products             20
Gas Station                 19
Alterations & Services      16
Shopping                    16
Automotive Accessories      1

In [54]:
# Quick patch for remaining unmapped labels
manual_patch = {
    'Stores': 'Retail',
    'Footwear': 'Retail',
    'Groceries': 'F&B',
    'Grocery': 'F&B',
    'Shopping': 'Retail',
    'Shops & Stores': 'Retail',
    'Retailers': 'Retail',
    'Retail Store': 'Retail',
    'Retail Stores': 'Retail',
    'Retail Trade': 'Retail',
    'Retail Shopping': 'Retail',
}

df_final['sector_level_1_clean'] = df_final['sector_level_1_clean'].replace(manual_patch)

# Recalculate sector score + KMF after patch
df_final['sector_score'] = df_final['sector_level_1_clean'].map(sector_score_map).fillna(6)
df_final['kmf_score'] = (
    df_final['reachability'] +
    df_final['sector_score'] +
    df_final['geo_score'] +
    df_final['data_completeness']
)
df_final['tier'] = df_final['kmf_score'].apply(assign_tier)

# Check nothing unmapped remains
still_unmapped = df_final[
    ~df_final['sector_level_1_clean'].isin(
        ['F&B','Beauty & Wellness','Retail',
         'Retail - Electronics','Professional Services','Others']
    ) & df_final['sector_level_1_clean'].notna()
]['sector_level_1_clean'].value_counts()

print(f"Remaining unmapped: {len(still_unmapped)}")
if len(still_unmapped) > 0:
    print(still_unmapped.head(10))

print(f"\n FINAL TIER BREAKDOWN:")
print(df_final.groupby('tier').agg(
    Count=('kmf_score','count'),
    Avg_KMF=('kmf_score','mean'),
).round(1))

print(f"\n FINAL SECTOR DISTRIBUTION:")
print(df_final['sector_level_1_clean'].value_counts())

df_final.to_csv("data/Cleaned_AU_Data.csv", index=False)
df_relevant = df_final.copy()
print(f"\n Final export complete: {len(df_final):,} rows | {len(df_final.columns)} columns")

Remaining unmapped: 20
sector_level_1_clean
Glassware & Earthenware    26
Office Products            20
Gas Station                19
Alterations & Services     16
Automotive Accessories     14
Nightlife                   6
Games                       6
Floor Mats                  5
Laundries,Services          5
Kitchen Equipment           4
Name: count, dtype: int64

 FINAL TIER BREAKDOWN:
        Count  Avg_KMF
tier                  
Tier 1  54853     79.0
Tier 2  67390     53.2
Tier 3  66546     31.5

 FINAL SECTOR DISTRIBUTION:
sector_level_1_clean
Others                        57990
Retail                        40611
Professional Services         35258
F&B                           27044
Beauty & Wellness             24855
Retail - Electronics           2895
Glassware & Earthenware          26
Office Products                  20
Gas Station                      19
Alterations & Services           16
Automotive Accessories           14
Nightlife                         6
Games    

In [55]:
# ── Final patch for remaining 20 unmapped labels ──
final_patch = {
    'Glassware & Earthenware': 'Retail',
    'Office Products': 'Retail - Electronics',
    'Gas Station': 'F&B',
    'Alterations & Services': 'Retail',
    'Automotive Accessories': 'Retail',
    'Nightlife': 'F&B',
    'Games': 'Retail - Electronics',
    'Floor Mats': 'Retail',
    'Laundries,Services': 'Retail',
    ' Laundries,Services': 'Retail',
    'Kitchen Equipment': 'Retail - Electronics',
    'Automotive Parts': 'Retail',
    'Automotive Repair': 'Professional Services',
    'Automotive Repair & Services': 'Professional Services',
    'Car Accessories': 'Retail',
    'Repair Services': 'Professional Services',
    'Printers': 'Retail - Electronics',
    'Pharmacy': 'Professional Services',
    'Equipment & Products': 'Professional Services',
    'Furniture & Supplies': 'Retail',
}

df_final['sector_level_1_clean'] = df_final['sector_level_1_clean'].replace(final_patch)

# Recalculate
df_final['sector_score'] = df_final['sector_level_1_clean'].map(sector_score_map).fillna(6)
df_final['kmf_score'] = (
    df_final['reachability'] +
    df_final['sector_score'] +
    df_final['geo_score'] +
    df_final['data_completeness']
)
df_final['tier'] = df_final['kmf_score'].apply(assign_tier)

# Verify nothing unmapped remains
still_unmapped = df_final[
    ~df_final['sector_level_1_clean'].isin(
        ['F&B','Beauty & Wellness','Retail',
         'Retail - Electronics','Professional Services','Others']
    ) & df_final['sector_level_1_clean'].notna()
]['sector_level_1_clean'].value_counts()

print(f"Remaining unmapped: {len(still_unmapped)}")
if len(still_unmapped) > 0:
    print(still_unmapped.to_string())

print(f"\n FINAL SECTOR DISTRIBUTION:")
for cat in ['F&B','Beauty & Wellness','Retail',
            'Retail - Electronics','Professional Services','Others']:
    count = (df_final['sector_level_1_clean'] == cat).sum()
    pct   = count / len(df_final) * 100
    print(f"   {cat:<25}: {count:>8,}  ({pct:.1f}%)")

print(f"\n FINAL TIER BREAKDOWN:")
tier_summary = df_final.groupby('tier').agg(
    Count=('kmf_score','count'),
    Avg_KMF=('kmf_score','mean'),
    Min_KMF=('kmf_score','min'),
    Max_KMF=('kmf_score','max')
).round(1)
print(tier_summary)

print(f"\n TIER 1 — TOP SECTORS:")
print(df_final[df_final['tier']=='Tier 1']['sector_level_1_clean'].value_counts())

print(f"\n TIER 1 — TOP SUBURBS:")
print(df_final[df_final['tier']=='Tier 1']['suburb'].value_counts().head(10))

df_final.to_csv("data/Cleaned_AU_Data.csv", index=False)
df_relevant = df_final.copy()
print(f"\n Complete: {len(df_final):,} rows | {len(df_final.columns)} columns")

Remaining unmapped: 9
sector_level_1_clean
Art Supplies                  2
Office Supplies & Services    2
Party Supplies                2
Telephones & Accessories      2
Video & DVD Equipment         1
Hobbies                       1
Automotive Products           1
Games & Accessories           1
Radios & Hi-Fi                1

 FINAL SECTOR DISTRIBUTION:
   F&B                      :   27,069  (14.3%)
   Beauty & Wellness        :   24,855  (13.2%)
   Retail                   :   40,677  (21.5%)
   Retail - Electronics     :    2,925  (1.5%)
   Professional Services    :   35,260  (18.7%)
   Others                   :   57,990  (30.7%)

 FINAL TIER BREAKDOWN:
        Count  Avg_KMF  Min_KMF  Max_KMF
tier                                    
Tier 1  54947     79.0     70.0    100.0
Tier 2  67296     53.2     40.0     69.0
Tier 3  66546     31.5     27.0     39.0

 TIER 1 — TOP SECTORS:
sector_level_1_clean
F&B                      23003
Retail                   19228
Beauty & Wellness

## 9. Rearrange the Tier 1 for easier outreach the merchants and BD works

In [56]:
# ── Raise Tier 1 threshold from 70 to 80 ──

def assign_tier_v2(score):
    if score >= 80:   return 'Tier 1'
    elif score >= 50: return 'Tier 2'
    else:             return 'Tier 3'

df_final['tier'] = df_final['kmf_score'].apply(assign_tier_v2)

print(" TIER BREAKDOWN (threshold adjusted to 80):")
tier_summary = df_final.groupby('tier').agg(
    Count=('kmf_score','count'),
    Pct=('kmf_score', lambda x: f"{len(x)/len(df_final)*100:.1f}%"),
    Avg_KMF=('kmf_score','mean'),
    Min_KMF=('kmf_score','min'),
    Max_KMF=('kmf_score','max')
).round(1)
print(tier_summary)

print(f"\n TIER 1 — TOP SECTORS:")
print(df_final[df_final['tier']=='Tier 1']['sector_level_1_clean'].value_counts())

print(f"\n TIER 1 — TOP SUBURBS:")
print(df_final[df_final['tier']=='Tier 1']['suburb'].value_counts().head(10))

df_final.to_csv("data/Cleaned_AU_Data.csv", index=False)
df_relevant = df_final.copy()
print(f"\n Final dataset exported with adjusted tiers")

 TIER BREAKDOWN (threshold adjusted to 80):
        Count    Pct  Avg_KMF  Min_KMF  Max_KMF
tier                                           
Tier 1  23700  12.6%     84.9     80.0    100.0
Tier 2  68447  36.3%     65.7     50.0     78.0
Tier 3  96642  51.2%     36.3     27.0     49.0

 TIER 1 — TOP SECTORS:
sector_level_1_clean
F&B                  19055
Retail                2685
Beauty & Wellness     1960
Name: count, dtype: int64

 TIER 1 — TOP SUBURBS:
suburb
sydney         2542
parramatta     1173
chatswood       768
bankstown       632
burwood         544
auburn          523
hurstville      465
haymarket       455
cabramatta      375
surry hills     345
Name: count, dtype: int64

 Final dataset exported with adjusted tiers


In [58]:
import os

# ── Create output directory if it doesn't exist ──
output_dir = "/Users/cindychen/iCloud Drive (Archive)/Desktop/👩‍💻Hardworker/Resume/Interview/Kpay_submission/data/Merchant priority"
os.makedirs(output_dir, exist_ok=True)

# ── Split by tier ──
tier1 = df_final[df_final['tier'] == 'Tier 1'].copy()
tier2 = df_final[df_final['tier'] == 'Tier 2'].copy()
tier3 = df_final[df_final['tier'] == 'Tier 3'].copy()

# ── Export ──
tier1.to_csv(f"{output_dir}/Tier_1_Merchants.csv", index=False)
tier2.to_csv(f"{output_dir}/Tier_2_Merchants.csv", index=False)
tier3.to_csv(f"{output_dir}/Tier_3_Merchants.csv", index=False)

print(f" Tier_1_Merchants.csv  →  {len(tier1):,} records")
print(f" Tier_2_Merchants.csv  →  {len(tier2):,} records")
print(f" Tier_3_Merchants.csv  →  {len(tier3):,} records")


 Tier_1_Merchants.csv  →  23,700 records
 Tier_2_Merchants.csv  →  68,447 records
 Tier_3_Merchants.csv  →  96,642 records


## Cleaned Done!